# P4 · Скор-колонка ПОСЛЕ log1p: что реально ест модель и как это дрейфует во времени

**Повод.** После фикса `--m_msg_transform log1p` результаты всё ещё выглядят подозрительно — «как будто что-то ломаем».

**Гипотезы:**
- **H1 (главная): log1p сжал масштаб, но не нестационарность.** `M` сбрасывается в ноль в начале **каждой** train-эпохи и растёт монотонно train→val→test (на eval сброса нет). Значит колонка каждую эпоху «разгоняется» 0 → ~6: ранние supervised-дни эпохи видят 0–2, поздние ~6, а eval всегда сидит на верхнем плато. Вес на колонке учится на смеси режимов, а читается на плато.
- **H2: масштаб всё ещё ×3–8.** log1p(400)≈6.0 против остальных фич ≤1 — мягче, чем ×400, но не нейтрально.
- **H3: что-то ещё в пути** (двойная трансформа, асимметрия каналов, баг карриера) — проверяется по ходу.

**План:** (1) что показывают текущие m-msg раны (кривые по эпохам); (2) пост-log1p распределения в обоих каналах **по дням стрима**, с симуляцией двух train-эпох (виден ли повторяющийся ramp) и продолжением в val/test; (3) сравнение масштабов с остальными фичами + сдвиг train→eval; (4) прототип стационаризирующей трансформы и вердикт.

In [3]:
# Ячейка 1 — кривые текущих ранов m-msg (genre): что именно «не нравится»
import numpy as np, polars as pl, torch, math, time
import plotly.express as px, plotly.graph_objects as go
import mlflow
from mlflow.tracking import MlflowClient
import sys; sys.path.insert(0, "..") if ".." not in sys.path else None
mlflow.set_tracking_uri("sqlite:///../mlruns/mlflow.db"); cl = MlflowClient()

rows = []
for expname in ["m-msg", "m-sampler"]:
    exp = mlflow.get_experiment_by_name(expname)
    if exp is None: continue
    for r in mlflow.search_runs(experiment_ids=[exp.experiment_id], output_format="list"):
        p = r.data.params
        if p.get("dataset") != "tgbn-genre" or p.get("global_hidden_dims") != "784": continue
        arm = f"{p.get('neighbor_sampler')}|mmsg={p.get('m_message_feature')}|tf={p.get('m_msg_transform')}|κ={p.get('kappa_select')}"
        for m in cl.get_metric_history(r.info.run_id, "val/ndcg"):
            rows.append((arm, int(m.step), m.value))
df = (pl.DataFrame(rows, schema=["arm", "epoch", "val"], orient="row")
        .group_by(["arm", "epoch"]).agg(pl.col("val").max()).sort(["arm", "epoch"]))  # max по дублям ранов
print(df.filter(pl.col("epoch") <= 10).pivot(values="val", index="epoch", on="arm").sort("epoch"))
fig = px.line(df.to_pandas(), x="epoch", y="val", color="arm", markers=True,
              title="genre d784: val NDCG@10 по эпохам (max по дублям)")
fig.update_layout(height=430, legend=dict(orientation="h", y=-0.3)); fig.show()

shape: (10, 6)
┌───────┬──────────────────┬─────────────────┬─────────────────┬─────────────────┬─────────────────┐
│ epoch ┆ memory|mmsg=None ┆ memory|mmsg=Tru ┆ memory|mmsg=Tru ┆ memory|mmsg=Tru ┆ recency|mmsg=Tr │
│ ---   ┆ |tf=None|κ=25.…  ┆ e|tf=None|κ=25. ┆ e|tf=None|κ=40. ┆ e|tf=log1p|κ=40 ┆ ue|tf=log1p|κ=4 │
│ i64   ┆ ---              ┆ …               ┆ …               ┆ …               ┆ …               │
│       ┆ f64              ┆ ---             ┆ ---             ┆ ---             ┆ ---             │
│       ┆                  ┆ f64             ┆ f64             ┆ f64             ┆ f64             │
╞═══════╪══════════════════╪═════════════════╪═════════════════╪═════════════════╪═════════════════╡
│ 1     ┆ 0.485738         ┆ 0.474171        ┆ 0.468788        ┆ 0.486521        ┆ 0.473772        │
│ 2     ┆ 0.49089          ┆ 0.486077        ┆ 0.487002        ┆ 0.492384        ┆ 0.480886        │
│ 3     ┆ 0.491709         ┆ 0.488689        ┆ 0.488515        ┆ 0.492205   

**Шаг 1 — паттерн поломки найден в кривых.** Все mmsg-руки (raw, log1p, recency+carrier) **пикуют на эпохе 2–3 и дальше монотонно деградируют** (log1p κ40: 0.4924@e2 → 0.4666@e10 — минус 0.026 за 8 эпох!), тогда как контроль без фичи стоит на плато ~0.488–0.492. Важно: log1p дал **лучший старт** (e2 бьёт контроль) — масштаб он починил, — но **не остановил спад**. Значит, дело не (только) в масштабе: это динамика обучения поверх **нестационарности** — чем дольше учимся, тем больше веса уезжает на информативную колонку, которая каждую эпоху проезжает одинаковый ramp (M сброшен → 0 → плато ~6), а eval всегда читает плато. Проверяем ramp и сдвиг train→eval напрямую.

In [4]:
# Ячейка 3 — post-log1p колонка по дням стрима: ramp каждой train-эпохи + плато на eval
from tgb.nodeproppred.dataset_pyg import PyGNodePropPredDataset
from torch_geometric.loader import TemporalDataLoader
from models.msampler import MSampler

ds = PyGNodePropPredDataset(name="tgbn-genre", root="datasets")
data = ds.get_TemporalData()
tr_d, va_d, te_d = data.train_val_test_split(val_ratio=0.15, test_ratio=0.15)
N, C = data.num_nodes, ds.num_classes
sw = torch.sort(tr_d.msg[:, 0]).values; alpha40 = math.log(2) / (40 * 86400.0)

def stream_scores(splits, smp, tag, day0=0):  # квантили log1p(score_for) на записываемых рёбрах по label-дням
    out, day = [], day0
    label_t = ds.get_label_time()
    for split, dd in splits:
        for b in TemporalDataLoader(dd, batch_size=200):
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                label_t = ds.get_label_time(); day += 1
                pm = b.t < float(lt[0][0])
                if pm.any():
                    sc = torch.log1p(smp.score_for(b.src[pm], b.dst[pm], b.t[pm]))
                    if day % 6 == 0:
                        out.append((tag, day, split, float(sc.median()), float(sc.quantile(0.95)), float((sc == 0).float().mean())))
                    smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
                m2 = ~pm
                if m2.any(): smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
            else:
                smp.insert(b.src, b.dst, b.t, b.msg)
    return out, day

rows = []
ds.reset_label_time()
r1, dmax = stream_scores([("train-эпоха1", tr_d)], MSampler(N, C, 10, alpha40, sw, device="cpu"), "e1")
ds.reset_label_time()  # ← симуляция начала СЛЕДУЮЩЕЙ эпохи: свежий M, тот же train (как делает train())
smp2 = MSampler(N, C, 10, alpha40, sw, device="cpu")
r2, _ = stream_scores([("train-эпоха2", tr_d)], smp2, "e2", day0=dmax)
ds.reset_label_time()
# после последней train-эпохи M НЕ сбрасывается и продолжает в val/test (как в реальном ране)
r3, _ = stream_scores([("val", va_d), ("test", te_d)], smp2, "e2", day0=2*dmax)
# ↑ ВАЖНО: r3 стримится тем же smp2, но лейбл-поинтер после reset прошёл train заново без записи —
# делаем честно: прокрутить train-лейблы без коллекции уже сделано внутри r2; здесь ds уже на val? проверим:
rows = r1 + r2 + r3
df2 = pl.DataFrame(rows, schema=["epoch", "day", "split", "log_med", "log_p95", "zero_frac"], orient="row")
print(df2.group_by("split").agg(pl.col("log_med").mean().round(3), pl.col("log_p95").mean().round(3),
                                pl.col("zero_frac").mean().round(3), pl.len()).sort("split"))

shape: (4, 5)
┌──────────────┬─────────┬─────────┬───────────┬─────┐
│ split        ┆ log_med ┆ log_p95 ┆ zero_frac ┆ len │
│ ---          ┆ ---     ┆ ---     ┆ ---       ┆ --- │
│ str          ┆ f64     ┆ f64     ┆ f64       ┆ u32 │
╞══════════════╪═════════╪═════════╪═══════════╪═════╡
│ test         ┆ 4.1     ┆ 6.368   ┆ 0.008     ┆ 23  │
│ train-эпоха1 ┆ 3.814   ┆ 6.015   ┆ 0.021     ┆ 199 │
│ train-эпоха2 ┆ 3.86    ┆ 6.03    ┆ 0.017     ┆ 198 │
│ val          ┆ 4.278   ┆ 6.725   ┆ 0.003     ┆ 21  │
└──────────────┴─────────┴─────────┴───────────┴─────┘


In [5]:
# Пила: log1p-медиана по дням двух train-эпох + плато val/test; доля ramp-режима
tr2 = df2.filter(pl.col("split").str.starts_with("train")).to_pandas()
vmed = float(df2.filter(pl.col("split") == "val")["log_med"].mean())
tmed = float(df2.filter(pl.col("split") == "test")["log_med"].mean())
fig = go.Figure()
fig.add_scatter(x=tr2["day"], y=tr2["log_med"], mode="lines", name="train log1p(score) медиана",
                line=dict(color="#2c7fb8"))
fig.add_vline(x=float(tr2[tr2["split"] == "train-эпоха2"]["day"].min()), line_dash="dash", line_color="black",
              annotation_text="reset M (эпоха 2)")
fig.add_hline(y=vmed, line_dash="dot", line_color="#d95f02", annotation_text=f"val плато ≈ {vmed:.2f}")
fig.add_hline(y=tmed, line_dash="dot", line_color="#d7191c", annotation_text=f"test плато ≈ {tmed:.2f}")
fig.update_layout(title="Пила нестационарности: каждую эпоху колонка проезжает ramp 0→плато, eval читает только плато",
                  xaxis_title="label-день (эпоха1 | эпоха2)", yaxis_title="log1p(score), медиана дня", height=430)
fig.show()
# квантификация ramp: доля train-дней с медианой заметно ниже eval-плато
for thr in [0.5, 0.75, 0.9]:
    frac = float((tr2["log_med"] < thr * vmed).mean())
    print(f"train-дней с медианой < {thr:.0%} val-плато: {frac:.1%}")
print(f"средний train (микс ramp+плато): {tr2['log_med'].mean():.2f} vs val {vmed:.2f} vs test {tmed:.2f}")

train-дней с медианой < 50% val-плато: 1.8%
train-дней с медианой < 75% val-плато: 15.6%
train-дней с медианой < 90% val-плато: 49.4%
средний train (микс ramp+плато): 3.84 vs val 4.28 vs test 4.10


In [6]:
# Ячейка 5 — прототип фикса: frozen train-ECDF скоров -> колонка стационарна по конструкции
# (1) реф: сабсэмпл скоров за один train-проход; (2) те же кривые в ECDF-пространстве
ref_vals = []
smp = MSampler(N, C, 10, alpha40, sw, device="cpu")
ds.reset_label_time(); label_t = ds.get_label_time(); day = 0
for b in TemporalDataLoader(tr_d, batch_size=200):
    if float(b.t[-1]) > label_t:
        lt = ds.get_node_label(b.t[-1])
        if lt is None: break
        label_t = ds.get_label_time(); day += 1
        pm = b.t < float(lt[0][0])
        if pm.any():
            if day % 3 == 0:  # сабсэмпл: каждый 3-й день
                ref_vals.append(smp.score_for(b.src[pm], b.dst[pm], b.t[pm]))
            smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
        m2 = ~pm
        if m2.any(): smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
    else:
        smp.insert(b.src, b.dst, b.t, b.msg)
ds.reset_label_time()
ref = torch.sort(torch.cat(ref_vals)).values
print(f"референс: {ref.numel():,} скоров, квантили [{ref[int(0.5*len(ref))]:.1f}, {ref[int(0.95*len(ref))]:.1f}, {ref.max():.0f}]")

def ecdf_tf(sc): return torch.searchsorted(ref, sc.contiguous(), right=True).float() / ref.numel()

# (2) — те же train-эпоха + val/test кривые, но колонка = ECDF(score)
def stream_ecdf(splits, smp, day0=0):
    out, day = [], day0
    label_t = ds.get_label_time()
    for split, dd in splits:
        for b in TemporalDataLoader(dd, batch_size=200):
            if float(b.t[-1]) > label_t:
                lt = ds.get_node_label(b.t[-1])
                if lt is None: break
                label_t = ds.get_label_time(); day += 1
                pm = b.t < float(lt[0][0])
                if pm.any():
                    sc = ecdf_tf(smp.score_for(b.src[pm], b.dst[pm], b.t[pm]))
                    if day % 6 == 0: out.append((day, split, float(sc.median()), float(sc.quantile(0.95))))
                    smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
                m2 = ~pm
                if m2.any(): smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
            else:
                smp.insert(b.src, b.dst, b.t, b.msg)
    return out, day
ds.reset_label_time()
s1, dmax = stream_ecdf([("train", tr_d)], MSampler(N, C, 10, alpha40, sw, device="cpu"))
s2, _ = stream_ecdf([("val", va_d), ("test", te_d)], smp := MSampler(N, C, 10, alpha40, sw, device="cpu"), day0=0) if False else ([], 0)
# val/test — продолжение train-состояния (как в реальном ране):
smp3 = MSampler(N, C, 10, alpha40, sw, device="cpu")
ds.reset_label_time(); _t, _ = stream_ecdf([("train", tr_d)], smp3); ds.reset_label_time()
s3, _ = stream_ecdf([("val", va_d), ("test", te_d)], smp3, day0=dmax)
de = pl.DataFrame(s1 + s3, schema=["day", "split", "ecdf_med", "ecdf_p95"], orient="row")
print(de.group_by("split").agg(pl.col("ecdf_med").mean().round(3), pl.col("ecdf_p95").mean().round(3), pl.len()).sort("split"))

референс: 39,842 скоров, квантили [46.1, 578.8, 5290]


shape: (3, 4)
┌───────┬──────────┬──────────┬─────┐
│ split ┆ ecdf_med ┆ ecdf_p95 ┆ len │
│ ---   ┆ ---      ┆ ---      ┆ --- │
│ str   ┆ f64      ┆ f64      ┆ u32 │
╞═══════╪══════════╪══════════╪═════╡
│ test  ┆ 0.584    ┆ 0.942    ┆ 27  │
│ train ┆ 0.499    ┆ 0.886    ┆ 199 │
│ val   ┆ 0.489    ┆ 0.867    ┆ 23  │
└───────┴──────────┴──────────┴─────┘


## Вердикт

**Что ломается:** не масштаб (его log1p починил — лучший старт, 0.4924@e2 > контроль), а **нестационарность колонки × динамика обучения**. Колонка каждую train-эпоху проезжает одинаковый ramp 0→плато (M сбрасывается; медиана train 3.84 при val-плато 4.28; половина train-дней ниже 90% плато), eval же всегда читает плато, причём рекуррентная память компаундит смещение через тысячи апдейтов val/test-стрима. Чем больше эпох — тем больше веса на колонке — тем глубже спад: 0.492@e2 → 0.467@e10, у ВСЕХ mmsg-рук, при плоском контроле.

**Фикс: `--m_msg_transform ecdf`** — замороженный train-ECDF скоров (реф собирается одним дешёвым пре-проходом по train, ~40k значений; зеркально φ-трансформе весов). Измерено на прототипе: колонка → Uniform[0,1] на train (медиана 0.499), **val 0.489 — сдвиг устранён**, test 0.584 (остаточный честный «больше истории»), масштаб ≤1. Прогноз: пик сохранится, монотонный спад после e2–3 исчезнет — это и есть falsifiable-тест фикса.

In [7]:
# Ячейка 7 — ФОРМА распределения скоров, что реально уходит в модель (оба канала), на val-плато
smp = MSampler(N, C, 10, alpha40, sw, device="cpu")
ds.reset_label_time(); label_t = ds.get_label_time()
for b in TemporalDataLoader(tr_d, batch_size=200):   # прогрев M на train (без сбора)
    while float(b.t[-1]) > label_t:
        if ds.get_node_label(b.t[-1]) is None: break
        label_t = ds.get_label_time()
    smp.insert(b.src, b.dst, b.t, b.msg)

mem_sc, gnn_sc = [], []                                # memory-канал (записываемые рёбра) и GNN-канал (top-k m_e_raw)
for b in TemporalDataLoader(va_d, batch_size=200):
    if float(b.t[-1]) > label_t:
        lt = ds.get_node_label(b.t[-1])
        if lt is None: break
        l0 = float(lt[0][0]); label_t = ds.get_label_time(); pm = b.t < l0
        if pm.any():
            mem_sc.append(smp.score_for(b.src[pm], b.dst[pm], b.t[pm]))
            out = smp(lt[1]); gnn_sc.append(out[4])       # m_e_raw = сырые скоры выбранных top-k рёбер
            smp.insert(b.src[pm], b.dst[pm], b.t[pm], b.msg[pm])
        m2 = ~pm
        if m2.any(): smp.insert(b.src[m2], b.dst[m2], b.t[m2], b.msg[m2])
    else:
        smp.insert(b.src, b.dst, b.t, b.msg)
ds.reset_label_time()
mem_sc = torch.cat(mem_sc); gnn_sc = torch.cat(gnn_sc)
print(f"memory-канал: {mem_sc.numel():,} скоров | GNN-канал: {gnn_sc.numel():,}")
for nm, x in [("memory (записываемые рёбра)", mem_sc), ("GNN (top-k выбранные)", gnn_sc)]:
    q = np.percentile(x.numpy(), [1, 25, 50, 75, 95, 99])
    frac0 = float((x <= 1.0).float().mean())        # доля «холодных» (скор ≤ 1, ~одно недавнее событие)
    print(f"{nm:30}: p1={q[0]:.2f} p25={q[1]:.2f} p50={q[2]:.2f} p75={q[3]:.1f} p95={q[4]:.0f} p99={q[5]:.0f} | доля(≤1)={frac0:.1%}")

memory-канал: 14,250 скоров | GNN-канал: 346,887
memory (записываемые рёбра)   : p1=0.00 p25=12.53 p50=52.91 p75=186.6 p95=909 p99=2318 | доля(≤1)=5.7%
GNN (top-k выбранные)         : p1=4.02 p25=21.99 p50=45.16 p75=94.1 p95=305 p99=694 | доля(≤1)=0.1%


In [8]:
from scipy import stats
from plotly.subplots import make_subplots

def sarle(x):  # bimodality coefficient: > 0.555 -> склонность к бимодальности (uniform=0.555)
    n = len(x); g = stats.skew(x); k = stats.kurtosis(x, fisher=True)
    return (g**2 + 1) / (k + 3*(n-1)**2/((n-2)*(n-3)))

fig = make_subplots(rows=1, cols=3, subplot_titles=["raw (log-x)", "log1p (что ест log1p-режим)", "ecdf (frozen train)"])
for nm, x, col in [("memory", mem_sc.numpy(), "#2c7fb8"), ("GNN top-k", gnn_sc.numpy(), "#d7191c")]:
    l1 = np.log1p(x); ec = ecdf_tf(torch.from_numpy(x)).numpy()
    fig.add_histogram(x=np.log10(np.clip(x, 1e-2, None)), name=nm, marker_color=col, opacity=0.6, nbinsx=60, row=1, col=1, legendgroup=nm, histnorm="probability density")
    fig.add_histogram(x=l1, name=nm, marker_color=col, opacity=0.6, nbinsx=60, row=1, col=2, legendgroup=nm, showlegend=False, histnorm="probability density")
    fig.add_histogram(x=ec, name=nm, marker_color=col, opacity=0.6, nbinsx=40, row=1, col=3, legendgroup=nm, showlegend=False, histnorm="probability density")
    print(f"{nm:10}: log1p → skew={stats.skew(l1):+.2f} kurt={stats.kurtosis(l1):+.2f} bimodality(Сарл)={sarle(l1):.3f}"
          + ("  ← БИМОДАЛЬНО (>0.555)" if sarle(l1) > 0.555 else "  (унимодально)"))
fig.update_layout(barmode="overlay", height=400, title="Распределение скор-колонки в трёх пространствах (плотность)", legend=dict(orientation="h", y=1.15))
fig.update_xaxes(title_text="log10(score)", row=1, col=1); fig.update_xaxes(title_text="log1p(score)", row=1, col=2); fig.update_xaxes(title_text="ECDF", row=1, col=3)
fig.show()

memory    : log1p → skew=-0.15 kurt=-0.57 bimodality(Сарл)=0.420  (унимодально)
GNN top-k : log1p → skew=+0.25 kurt=+0.16 bimodality(Сарл)=0.337  (унимодально)


## Форма распределения скоров

**Ни нормальное, ни бимодальное — тяжёлохвостое ~лог-нормальное в raw, унимодальное после log.**

- **raw**: правый тяжёлый хвост через 5 порядков (0.01 → 2318), классический лог-нормальный «взрыв счётчиков». В memory-канале есть небольшой **холодный спайк** (5.7% скоров ≤ 1 — пары, увиденные ~раз недавно); в GNN-канале его почти нет (0.1%) — top-k-выборка сама срезает холодные ячейки, оставляя правую часть.
- **log1p**: лог де-скьюит хвост в **унимодальную** форму. Коэффициент Сарла **0.42 (memory) / 0.34 (GNN)** — оба **ниже порога бимодальности 0.555**, т.е. один горб, не два. memory при этом **платикуртичен** (kurt −0.57 → широкое плоское плато, ближе к равномерному), GNN — почти нормальный (skew +0.25).
- **ecdf**: по конструкции → Uniform[0,1] на train; на val горб чуть уезжает вправо (p50 0.499→0.584 train→test) — это и есть остаточный дрейф, который ecdf убирает из **положения**, оставляя форму.

**Вывод для фикса.** Проблема была не в форме (бимодальности нет — иначе GRU-гейт вообще нельзя было бы линейно откалибровать). Проблема — в **дрейфе положения** унимодального горба (нестационарность из предыдущего шага) + остаточном масштабе. Раз лог-форма унимодальна и монотонна, **ecdf-отображение чистое** (без разрывов от второго горба) и корректно фиксирует положение в Uniform[0,1]. Т.е. диагноз согласован: `ecdf` — правильная трансформа.

In [9]:
# Ячейка 10 — честное сравнение: контроль тоже падает (общий оверфит), log1p ≥ контроль по best-val
def curves(arm_filter):
    out = {}
    for expname in ["m-msg", "m-sampler"]:
        e = mlflow.get_experiment_by_name(expname)
        if e is None: continue
        for r in mlflow.search_runs(experiment_ids=[e.experiment_id], output_format="list"):
            p = r.data.params
            if p.get("dataset") != "tgbn-genre" or p.get("global_hidden_dims") != "784": continue
            mm, tf = p.get("m_message_feature"), p.get("m_msg_transform")
            key = "off" if mm != "True" else f"mmsg-{tf}"
            if key not in arm_filter or p.get("neighbor_sampler") != "memory": continue
            vh = {int(m.step): m.value for m in cl.get_metric_history(r.info.run_id, "val/ndcg")}
            th = {int(m.step): m.value for m in cl.get_metric_history(r.info.run_id, "test/ndcg")}
            if vh and (key not in out or max(vh) > max(out[key][0])): out[key] = (vh, th)
    return out
cv = curves({"off", "mmsg-log1p", "mmsg-None"})
fig = go.Figure()
name_map = {"off": "контроль (без mmsg)", "mmsg-None": "mmsg raw", "mmsg-log1p": "mmsg log1p"}
for key, (vh, th) in cv.items():
    eps = sorted(vh); be = max(vh, key=lambda x: vh[x])
    fig.add_scatter(x=eps, y=[vh[e] for e in eps], mode="lines+markers", name=name_map.get(key, key))
    fig.add_scatter(x=[be], y=[vh[be]], mode="markers", marker=dict(size=13, symbol="star"), showlegend=False)
    print(f"{name_map.get(key,key):22}: best_val={vh[be]:.4f}@e{be}  test@best={th.get(be,float('nan')):.4f}  (эпох {max(eps)})")
fig.update_layout(title="genre d784: ВСЕ руки (и контроль) пикуют @e2-4 и падают — оверфит, не фича. ★=best-val",
                  xaxis_title="эпоха", yaxis_title="val NDCG@10", height=430)
fig.show()

mmsg log1p            : best_val=0.4924@e2  test@best=0.4895  (эпох 16)
mmsg raw              : best_val=0.4888@e4  test@best=0.4821  (эпох 7)
контроль (без mmsg)   : best_val=0.4890@e3  test@best=0.4835  (эпох 5)


## Итог «глубокого копания»: мы чинили не то

По **test@best-val** (число для статьи) на genre: **mmsg log1p 0.4895 > контроль 0.4835 > mmsg raw 0.4821**. То есть:
- **«mmsg вредит» относилось к RAW-руке** (сатурация гейтов, 0.482) — log1p её починил и вывел в **+0.006 над контролем**, а не в минус. Впечатление «хуже» шло от сравнения по last-epoch падающей кривой либо от raw-рана.
- **Спад пик@e2→0.43@e16 — общий оверфит d=784**: контроль без mmsg падает так же (−0.0029/эп). best-val его снимает → на итоговое число он не влияет.
- **Самокоррекция по ecdf**: стационаризация целила в *спад*, но спад **не портит best-val** (он и так берёт пик до оверфита). Поэтому ecdf и «не помог» — он решает проблему, которой нет в метрике-что-важна. Плюс ecdf-раны недокручены (genre 2 эпохи, trade 26). log1p при этом остаётся дефолтным рабочим вариантом.

**Что реально осталось выяснить** (а не очередная трансформа): **+0.006 log1p над контролем — это сигнал или шум одного сида?** Оверфит d=784 + best-val-на-падающей-кривой + один сид (шум ±0.003) делают число хрупким. Чистый тест — в режиме **d=128, где оверфита нет** (distill-iter: монотонный рост 20 эпох до 0.49): matched контроль vs log1p vs ecdf, несколько сидов. Это и отделит «фича добавляет сигнал» от «d=784 оверфитит».